# Ask 6 — Grounded or Invented

Fluency is constant; only the grounding check tells a supported answer
from a confident invention. The checker runs offline; a model version of
it runs with the class key.

In [ ]:
# The pile: documents from the (fictional) Jefferson High School.
# Real enough to search, small enough to read whole.
PILE = {
 "handbook_academics": """S4.1 Grading scale. A 90-100, B 80-89, C 70-79, D 60-69.
Semester grades weight exams at 30 percent.
S4.2 Exam Retake Policy. This policy applies to final exams only. Students
receive one retake per semester, requested within ten school days. The
higher score stands.
S4.3 Grade appeals. Appeals go to the department head in writing within
fifteen school days of the posted grade.
S4.5 Late work. Assignments lose 10 percent per school day late, to a
maximum of 50 percent. Teachers may grant extensions for documented
emergencies.""",
 "handbook_schedule": """S2.0 Bell schedule. Regular days run eight periods,
8:15 AM to 3:20 PM.
S2.1 Wednesday schedule. Dismissal at 1:30 PM every Wednesday for staff
development.
S2.4 Late arrival. Students arriving after 8:30 AM sign in at the main
office with a note.""",
 "handbook_trips": """S5.1 Field trips require a signed permission form
submitted five school days in advance.
S5.2 Trip costs above 20 dollars qualify for the student activity fund.
S5.4 Chaperones must be approved district volunteers.""",
 "handbook_athletics": """S6.2 Eligibility. Athletes must hold a C average
during their season. Freshmen may try out for varsity teams.
S6.3 Petitions. A varsity roster spot for a freshman requires a coach's
petition to the athletic director.""",
 "robotics_minutes": """Robotics club meets Tuesdays in room 214. Regional
trip is April 18; bring your signed permission form by April 10. Dues are
15 dollars for the year.""",
 "clubs_list": """Active clubs: robotics (Tuesdays), debate (Thursdays),
art collective (Fridays), chess (lunch, library). Sign-up forms at the
student office.""",
 "bus_routes": """Routes 12 and 15 serve the north side. Final pickup at
4:45 PM outside door C. Activity buses run Tuesday and Thursday only.""",
 "cafeteria": """Lunch periods run 11:10, 11:55, and 12:40. Breakfast is
served from 7:40 AM. Menus post monthly on the food services page.""",
}
print(f"{len(PILE)} documents, {sum(len(t) for t in PILE.values())} characters total")

In [ ]:
def chunk_by_section(pile, overlap_sentences=1):
    """Cut on the S-section seams; carry a sentence of overlap across cuts."""
    chunks = []
    for doc, text in pile.items():
        parts, current, header = [], [], None
        for line in text.splitlines():
            if line.strip().startswith("S") and len(line) > 2 and line.strip()[1].isdigit():
                if current:
                    parts.append((header, " ".join(current)))
                header, current = line.strip().split()[0].rstrip("."), [line]
            else:
                current.append(line)
        if current:
            parts.append((header, " ".join(current)))
        for i, (header, body) in enumerate(parts):
            text_out = body
            if overlap_sentences and i > 0:
                prev_tail = parts[i-1][1].split(". ")[-1]
                text_out = prev_tail + " ... " + body
            chunks.append({"doc": doc, "section": header or doc, "text": " ".join(text_out.split())})
    return chunks

CHUNKS = chunk_by_section(PILE)
print(f"{len(CHUNKS)} chunks")
for c in CHUNKS[:3]:
    print(f"  [{c['doc']} {c['section']}] {c['text'][:70]}...")

In [ ]:
%pip install -q anthropic

In [ ]:
import os, getpass
# Ask your teacher for the class API key. getpass keeps it out of the file.
try:
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Class API key: ")
    HAVE_KEY = len(os.environ["ANTHROPIC_API_KEY"]) > 10
except Exception:
    HAVE_KEY = False
print("Key loaded." if HAVE_KEY else "No key - precomputed outputs shown below each live cell.")

In [ ]:
MODEL = "claude-opus-5"

def llm(prompt, max_tokens=800):
    import anthropic
    client = anthropic.Anthropic()
    return client.messages.create(model=MODEL, max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}]).content[-1].text

## A sabotaged run

We hand the model the WRONG chunks on purpose — the near-miss case, close
enough that it answers instead of refusing. (Precomputed; with a key you
can reproduce it by retrieving with the wrong query.)

In [ ]:
provided = [c for c in CHUNKS if c["section"] in ("S5.4", "S2.0")]
answer_b = ("Permission forms are typically due one week before the trip. "
            "Chaperones must be approved district volunteers [S5.4].")
print("chunks provided:", [c["section"] for c in provided])
print("answer:", answer_b)

## The grounding check, mechanical version

Claim by claim: do the claim's meaningful words and numbers appear in ANY
provided chunk? Absent → invented, no matter how reasonable — and "true
but ungrounded" still fails, because your system can't tell it from
fiction.

In [ ]:
import re

def ground_check(answer, provided_chunks):
    ctx = " ".join(c["text"].lower() for c in provided_chunks)
    verdicts = []
    for sent in re.split(r"(?<=[.!?]) ", answer):
        content = [w for w in re.findall(r"[a-z0-9]+", sent.lower())
                   if len(w) > 3 and w not in ("must", "with", "that", "this", "typically", "before")]
        found = [w for w in content if w in ctx]
        ratio = len(found) / len(content) if content else 1.0
        verdicts.append((ratio > 0.6, round(ratio, 2), sent.strip()))
    return verdicts

for ok, ratio, sent in ground_check(answer_b, provided):
    print(("GROUNDED " if ok else "INVENTED "), f"({ratio:4.0%} of content words in chunks)  {sent}")

flags = [ok for ok, r, s in ground_check(answer_b, provided)]
assert flags == [False, True], "sentence 1 is invented, sentence 2 is grounded"

The invented sentence is *plausible* — most schools do want forms
about a week early; Jefferson's real rule (S5.1, which retrieval failed to
deliver) says five school days. Plausible-and-ungrounded is exactly the
case the checker exists for.

## The model as checker

A model checking text against text is reading, not remembering — solid
ground. Same adversarial stance as the mechanical version:

In [ ]:
CHECK_PROMPT = """Here are the provided sections and an answer. List every claim
in the answer that the sections do NOT support. Reply as JSON:
{{"unsupported": ["<claim>", ...]}}

SECTIONS:
{sections}

ANSWER: {answer}"""

if HAVE_KEY:
    sections = "\n".join(f"[{c['section']}] {c['text']}" for c in provided)
    print(llm(CHECK_PROMPT.format(sections=sections, answer=answer_b)))
else:
    print('Precomputed: {"unsupported": ["Permission forms are typically due')
    print('one week before the trip"]}')
    print()
    print("Same verdict as the mechanical checker - and on subtler cases the")
    print("model version wins, because 'support' is finally about meaning.")

## Try it

1. Run `ground_check` on lesson 5's GOOD answer with its real chunks —
   confirm all sentences pass.
2. Write an answer that is TRUE (check the pile) but cites nothing the
   provided chunks say. Watch it fail — then write the sentence explaining
   to a classmate why that's the check working, not a bug.
3. **Build turn-in:** claim-level audit of your ten answers, each invention
   traced to its cause: retrieval delivered wrong chunks, or the model
   ignored the contract.